# Part 1 Data Understanding Quality Statistics

Người 1 — Data Understanding, Data Quality, Descriptive Statistics, Normalization và kiểm định phân phối.

**Lưu ý:** Các cell nền gốc 2–14 được sao chép để notebook phần này có các biến dùng chung. Khi tích hợp cuối, chỉ giữ cell nền một lần; lấy phần riêng của từng người theo README.

In [1]:
from pathlib import Path
import json
import hashlib
import warnings
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter
from scipy import stats
from IPython.display import display, Markdown

DATA_PATH = ""  # optional: exact file path, for example /content/drive/MyDrive/.../listings_processed.xlsx
OUTPUT_DIR = Path("report34_outputs")
FIG_DIR = OUTPUT_DIR / "figures"
TABLE_DIR = OUTPUT_DIR / "tables"
RANDOM_SEED = 42
MIN_GROUP_N = 8
# Notebook cần ba sheet sau; thứ tự hiển thị trong Excel không quan trọng.
REQUIRED_SHEETS = ("Listings", "MacroLookup", "ChangeLog")
RNG = np.random.default_rng(RANDOM_SEED)
FIG_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)
# Mỗi lần Run All chỉ xóa các bằng chứng do notebook này tự tạo trong output của nó.
# Điều này ngăn các biểu đồ từ lần chạy cũ vượt qua bước kiểm tra số lượng hình.
for old_figure in FIG_DIR.glob("H*.png"):
    old_figure.unlink()
for old_table in TABLE_DIR.glob("*.csv"):
    old_table.unlink()
old_summary = OUTPUT_DIR / "summary_real.json"
if old_summary.exists():
    old_summary.unlink()
plt.rcParams.update({
    "figure.dpi": 120, "savefig.dpi": 190,
    "font.family": "DejaVu Sans", "font.size": 10,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.titleweight": "bold", "grid.alpha": 0.22,
})

def locate_input(path):
    options = ([Path(path).expanduser()] if str(path).strip() else []) + [
        Path("listings_processed.xlsx"),
        Path("data/listings_processed.xlsx"),
        Path("/content/listings_processed.xlsx"),
        Path("/mnt/data/listings_processed.xlsx"),
    ]
    for p in options:
        if p.is_file():
            return p.resolve()
    raise FileNotFoundError(
        "Không tìm thấy listings_processed.xlsx. Hãy upload file hoặc cập nhật DATA_PATH."
    )

SOURCE = locate_input(DATA_PATH)
SHA256 = hashlib.sha256(SOURCE.read_bytes()).hexdigest()
print("Data source:", SOURCE)
print("SHA-256:", SHA256)
print("Output folder:", OUTPUT_DIR.resolve())

Data source: /mnt/data/ady_humanized_verify/listings_processed.xlsx
SHA-256: b6a3afa73103d55959eba36ec21349a23bdd1bcddeaea353ea927e2b7ac5a051
Output folder: /mnt/data/ady_humanized_verify/report34_outputs


In [2]:
# Đọc nguyên trạng ba sheet; không lọc/mất dòng âm thầm.
raw = pd.read_excel(SOURCE, sheet_name="Listings")
macro = pd.read_excel(SOURCE, sheet_name="MacroLookup")
log = pd.read_excel(SOURCE, sheet_name="ChangeLog")
assert all(name in pd.ExcelFile(SOURCE).sheet_names for name in REQUIRED_SHEETS), "Thiếu sheet nguồn"
required = {"listing_code", "price_vnd_million", "brand", "province",
            "condition", "year", "scraped_at", "mileage_km"}
missing_cols = required - set(raw.columns)
if missing_cols:
    raise ValueError(f"Thiếu cột bắt buộc: {sorted(missing_cols)}")
print("Listings:", raw.shape, "| MacroLookup:", macro.shape, "| ChangeLog:", log.shape)
display(raw.head(3))
display(macro.head(3))
display(log.head(3))

Listings: (632, 20) | MacroLookup: (12, 4) | ChangeLog: (265, 5)


,province,brand,title,year,price_vnd_million,mileage_km,location_text,listing_code,detail_url,scraped_at,condition,model_trim,car_classification,dan_so,gdp_binh_quan_dau_nguoi_trieu_dong,ty_le_do_thi_hoa,fuel_type,vehicle_age,price_to_gdp_ratio,urbanization_group
0,Hà Nội,Toyota,[Xe mới] Toyota Innova Cross HEV 2.0 CVT (2026),2026,900,NaN,Hà Nội,7001476,https://bonbanh.com/xe-toyota-innova-cross-hev...,2026-10-07T10:36:30,mới,Toyota Innova Cross HEV 2.0 CVT,Xe mới,8587100,150,0.49,Hybrid,0,6.00,Trung bình (30-70%)
1,Hà Nội,Toyota,[Xe cũ] Toyota Prado TXL 2.7L (2016),2016,1179,120000.0,Hà Nội,7017800,https://bonbanh.com/xe-toyota-prado-txl-2.7l-2...,2026-10-07T10:36:30,cũ,Toyota Prado TXL 2.7L,Xe cũ,8587100,150,0.49,Petrol,10,7.86,Trung bình (30-70%)
2,Hà Nội,Toyota,[Xe cũ] Toyota Innova 2.0E (2020),2020,435,100000.0,Hà Nội,7022277,https://bonbanh.com/xe-toyota-innova-2.0e-2020...,2026-10-07T10:36:30,cũ,Toyota Innova 2.0E,Xe cũ,8587100,150,0.49,Petrol,6,2.90,Trung bình (30-70%)


,tinh_thanh,dan_so,gdp_binh_quan_dau_nguoi_trieu_dong,ty_le_do_thi_hoa
0,Bà Rịa - Vũng Tàu,1192900,350,0.60
1,Bình Dương,2763100,175,0.84
2,Cần Thơ,1251800,100,0.73


,listing_code,cot,truoc,sau,ly_do
0,NaN,(cả dòng),Giá xe Hyundai Grand i10 2026,ĐÃ LOẠI,Không phải tin đăng
1,7001476.0,(cả dòng),[Xe mới] Toyota Innova Cross HEV 2.0 CVT (2026),ĐÃ LOẠI,Trùng listing_code (bản sao)
2,7017800.0,(cả dòng),[Xe cũ] Toyota Prado TXL 2.7L (2016),ĐÃ LOẠI,Trùng listing_code (bản sao)


# REPORT 3 — Tìm hiểu dữ liệu, thống kê và xử lý dữ liệu

## 1. Data Understanding — nguồn, grain, thang đo

Dữ liệu có tính **cross-sectional** (các tin đăng thu thập trong khoảng ngắn), `scraped_at` là mốc thu thập chứ không phải lịch sử giá của cùng xe. `brand`, `province`, `fuel_type` là **nominal**; `price_vnd_million`, `mileage_km`, `vehicle_age` là **ratio**, `condition` là nhãn phân loại. `year` là **interval** theo slide 6.1 (khoảng cách năm có ý nghĩa; 0 không phải điểm gốc tự nhiên); `vehicle_age` là **ratio** dùng trong phân tích tương quan, không đưa cùng `year` vào ma trận vì `vehicle_age = năm thu thập − year`; `listing_code` là mã định danh (không lấy trung bình). `price_to_gdp_ratio` là cột dẫn xuất, không phải thước đo khả năng chi trả hộ gia đình.

Bảng `MacroLookup` mang cấp độ **tỉnh/thành**. Thông tin vĩ mô lặp lại trên mỗi tin đăng **không tạo ra thêm số quan sát tỉnh độc lập**; nguồn/năm đo của các chỉ số macro phải xác minh bên ngoài trước khi coi là thống kê chính thức.

**Kiểm định hợp lệ dữ liệu đầu vào:** phân biệt giá trị rỗng thực (`NaN`) và nhãn phân loại `Unknown`; `vehicle_age` được dẫn xuất từ năm cào dữ liệu và năm sản xuất, vì vậy không cùng đưa `year` và `vehicle_age` vào phân tích tương quan đa biến.


In [3]:
d = raw.copy(deep=True)
for c in ["price_vnd_million", "mileage_km", "vehicle_age", "year",
          "gdp_binh_quan_dau_nguoi_trieu_dong", "ty_le_do_thi_hoa", "dan_so",
          "price_to_gdp_ratio"]:
    if c in d.columns:
        d[c] = pd.to_numeric(d[c], errors="coerce")
d["scraped_at"] = pd.to_datetime(d["scraped_at"], errors="coerce")
d["scrape_date"] = d["scraped_at"].dt.date
if "vehicle_age" not in d.columns:
    d["vehicle_age"] = (d["scraped_at"].dt.year - d["year"]).clip(lower=0)

variable_dictionary = pd.DataFrame([
    ["listing_code", "ID", "Tin đăng", "Khóa định danh, không tính trung bình"],
    ["price_vnd_million", "Ratio", "Triệu VND", "Giá rao bán, biến phân tích chính"],
    ["brand", "Nominal", "Tên hãng", "So sánh cơ cấu mẫu"],
    ["province", "Nominal", "Địa phương", "5 tỉnh được lấy mẫu"],
    ["condition", "Nominal", "Mới/cũ", "Cần phân tầng trong phân tích"],
    ["mileage_km", "Ratio", "km", "Thiếu có tính cấu trúc với xe mới"],
    ["year", "Interval", "Năm lịch", "Khoảng cách năm có ý nghĩa, năm 0 không là mốc ratio"],
    ["vehicle_age", "Ratio", "Năm tại lúc cào", "Dẫn xuất từ năm cào trừ năm xe; không đưa đồng thời với year vào correlation"],
    ["fuel_type", "Nominal", "Loại nhiên liệu", "'Unknown' không tương đương NULL"],
    ["scraped_at", "Datetime", "Thời điểm", "Không suy ra xu hướng từ 1 ngày"],
], columns=["Biến", "Thang đo", "Đơn vị", "Vai trò / lưu ý"])
display(variable_dictionary)
variable_dictionary.to_csv(TABLE_DIR / "01_variable_dictionary.csv", index=False, encoding="utf-8-sig")

,Biến,Thang đo,Đơn vị,Vai trò / lưu ý
0,listing_code,ID,Tin đăng,"Khóa định danh, không tính trung bình"
1,price_vnd_million,Ratio,Triệu VND,"Giá rao bán, biến phân tích chính"
2,brand,Nominal,Tên hãng,So sánh cơ cấu mẫu
3,province,Nominal,Địa phương,5 tỉnh được lấy mẫu
4,condition,Nominal,Mới/cũ,Cần phân tầng trong phân tích
5,mileage_km,Ratio,km,Thiếu có tính cấu trúc với xe mới
6,year,Interval,Năm lịch,"Khoảng cách năm có ý nghĩa, năm 0 không là mốc..."
7,vehicle_age,Ratio,Năm tại lúc cào,Dẫn xuất từ năm cào trừ năm xe; không đưa đồng...
8,fuel_type,Nominal,Loại nhiên liệu,'Unknown' không tương đương NULL
9,scraped_at,Datetime,Thời điểm,Không suy ra xu hướng từ 1 ngày


## 2. Data Quality — thiếu, trùng, phạm vi và kiểm tra bất thường

Không tự động loại giá cao theo IQR: xe sang/hiếm có thể là quan sát hợp lệ. Chỉ loại **khỏi phép phân tích giá** các dòng giá không hữu hạn/không dương; không sửa file nguồn. **Không điền `mileage_km` thiếu bằng 0** vì xe cũ cũng thiếu và 0 km có ý nghĩa khác NULL. Giữ log để truy vết thao tác xử lý ở pipeline trước.

**Quy tắc xử lý:** không tự loại 40 tin nghi trùng gần; giữ mẫu gốc 632 và đối chiếu độ nhạy trên mẫu chỉ giữ mỗi nhóm 1 tin. `fuel_type=Unknown` được giữ như trạng thái chưa xác minh; nếu so sánh nhiên liệu thực sự thì loại nhóm Unknown khỏi mẫu của phép so sánh đó và báo rõ n. 80 xe mới thiếu km là thiếu theo cấu trúc, 138 xe cũ thiếu km phải kiểm tra sai lệch. Giá trị km làm tròn/nhỏ bất thường được gắn cờ để kiểm tra thủ công, không tự sửa.

In [4]:
quality = pd.DataFrame({
    "missing_n": d.isna().sum(),
    "missing_pct": (d.isna().mean() * 100).round(2),
    "dtype": d.dtypes.astype(str),
    "unique_n": d.nunique(dropna=True),
}).sort_values("missing_n", ascending=False)
flags = pd.DataFrame(index=d.index)
flags["missing_or_nonpositive_price"] = d["price_vnd_million"].isna() | (d["price_vnd_million"] <= 0)
flags["negative_mileage"] = d["mileage_km"].notna() & (d["mileage_km"] < 0)
flags["invalid_vehicle_age"] = d["vehicle_age"].isna() | (d["vehicle_age"] < 0)
flags["repeated_listing_code"] = d["listing_code"].duplicated(keep=False)
flags["year_after_scrape"] = d["year"] > d["scraped_at"].dt.year
flags["vehicle_age_inconsistent"] = (d["vehicle_age"].notna() & d["year"].notna()
    & d["scraped_at"].notna() &
    ((d["scraped_at"].dt.year - d["year"]) != d["vehicle_age"]))
flags["missing_scraped_at"] = d["scraped_at"].isna()
assert d["listing_code"].notna().all(), "Cần kiểm tra mã tin đăng NULL"
assert not flags["repeated_listing_code"].any(), "Có listing_code trùng trong Listings"
a = d.loc[~flags["missing_or_nonpositive_price"]].copy()
missing_mileage_by_condition = d.groupby("condition")["mileage_km"].agg(
    n="size", missing=lambda x: int(x.isna().sum()),
    missing_pct=lambda x: round(x.isna().mean()*100, 2))
print("FLAG COUNTS:")
display(flags.sum().to_frame("n_flags"))
print("MISSING PROFILE:")
display(quality)
print("ODOMETER MISSING BY CONDITION:")
display(missing_mileage_by_condition)
print("DATES:", d["scrape_date"].value_counts(dropna=False).to_dict())
print("PROVINCES:", sorted(d["province"].dropna().unique()))
quality.to_csv(TABLE_DIR / "02_quality.csv", encoding="utf-8-sig")
flags.sum().to_csv(TABLE_DIR / "03_flags.csv", encoding="utf-8-sig")
missing_mileage_by_condition.to_csv(TABLE_DIR / "04_missing_by_condition.csv", encoding="utf-8-sig")

# Kiểm tra tin gần trùng: title + price + province, không xóa tự động.
NEAR_KEYS = ["title", "price_vnd_million", "province"]
near_mask = d.duplicated(NEAR_KEYS, keep=False)
near_group_count = int(d.loc[near_mask].groupby(NEAR_KEYS, dropna=False).ngroups)
near_rows = int(near_mask.sum())
near_excess = int(d.duplicated(NEAR_KEYS).sum())
near_detail = d.loc[near_mask, ["listing_code"] + NEAR_KEYS + ["condition", "mileage_km"]].sort_values(NEAR_KEYS)
near_detail.to_csv(TABLE_DIR / "05_near_duplicate_candidates.csv", index=False, encoding="utf-8-sig")
print(f"TIN NGHI TRÙNG GẦN: {near_rows} dòng / {near_group_count} nhóm / {near_excess} dòng dư nếu giữ 1 mỗi nhóm")
display(near_detail.head(10))

# Kiểm tra thủ công cần lưu ý; không tự suy đoán km có lỗi.
used_quality = d.loc[d["condition"].eq("cũ")].copy()
suspicious_km = used_quality.loc[used_quality["mileage_km"].notna() & (used_quality["mileage_km"] < 1000),
    ["listing_code", "title", "year", "mileage_km", "price_vnd_million"]]
suspicious_km.to_csv(TABLE_DIR / "06_low_mileage_review.csv", index=False, encoding="utf-8-sig")
print("Xe cũ có km < 1.000:",len(suspicious_km))
print("Mốc km được khai báo phổ biến:")
display(used_quality["mileage_km"].value_counts().head(10).rename_axis("km").to_frame("Số tin"))


FLAG COUNTS:


,n_flags
missing_or_nonpositive_price,0
negative_mileage,0
invalid_vehicle_age,0
repeated_listing_code,0
year_after_scrape,0
vehicle_age_inconsistent,0
missing_scraped_at,0


MISSING PROFILE:


,missing_n,missing_pct,dtype,unique_n
mileage_km,218,34.49,float64,137
brand,0,0.00,object,7
title,0,0.00,object,481
year,0,0.00,int64,26
province,0,0.00,object,5
price_vnd_million,0,0.00,int64,298
location_text,0,0.00,object,5
listing_code,0,0.00,int64,632
detail_url,0,0.00,object,632
scraped_at,0,0.00,datetime64[ns],37


ODOMETER MISSING BY CONDITION:


,n,missing,missing_pct
condition,,,
cũ,552,138,25.0
mới,80,80,100.0


DATES: {datetime.date(2026, 10, 7): 632}
PROVINCES: ['Cần Thơ', 'Hà Nội', 'Nghệ An', 'Quảng Ninh', 'Đồng Nai']
TIN NGHI TRÙNG GẦN: 40 dòng / 19 nhóm / 21 dòng dư nếu giữ 1 mỗi nhóm


,listing_code,title,price_vnd_million,province,condition,mileage_km
358,6785533,[Xe cũ] Ford Ranger Raptor 2.0L 4x4 AT (2024),1050,Quảng Ninh,cũ,10000.0
370,7003608,[Xe cũ] Ford Ranger Raptor 2.0L 4x4 AT (2024),1050,Quảng Ninh,cũ,50000.0
245,6945154,[Xe cũ] Ford Ranger XLS 2.2L 4x2 AT (2017),379,Đồng Nai,cũ,90000.0
254,6945171,[Xe cũ] Ford Ranger XLS 2.2L 4x2 AT (2017),379,Đồng Nai,cũ,90000.0
489,7019913,[Xe cũ] Ford Ranger XLS 2.2L 4x2 AT (2020),455,Nghệ An,cũ,NaN
492,6729892,[Xe cũ] Ford Ranger XLS 2.2L 4x2 AT (2020),455,Nghệ An,cũ,100000.0
203,6872574,[Xe cũ] Ford Territory Titanium X 1.5 AT (2024),730,Hà Nội,cũ,21000.0
209,6913460,[Xe cũ] Ford Territory Titanium X 1.5 AT (2024),730,Hà Nội,cũ,28000.0
302,6872343,[Xe cũ] Hyundai SantaFe Cao cấp 2.2L HTRAC (2021),930,Quảng Ninh,cũ,80000.0
314,6735964,[Xe cũ] Hyundai SantaFe Cao cấp 2.2L HTRAC (2021),930,Quảng Ninh,cũ,80000.0


Xe cũ có km < 1.000: 5
Mốc km được khai báo phổ biến:


,Số tin
km,
70000.0,23
60000.0,22
50000.0,21
100000.0,20
90000.0,18
80000.0,16
40000.0,15
110000.0,12
30000.0,11


In [5]:
# Đối soát phép tính/khóa nối trong file xử lý; không khẳng định tính chính xác số liệu vĩ mô.
print("Listings ID unique:", a["listing_code"].is_unique)
print("Các địa phương thuộc MacroLookup:", set(a["province"]).issubset(set(macro["tinh_thanh"])))
if {"gdp_binh_quan_dau_nguoi_trieu_dong", "price_to_gdp_ratio"}.issubset(a.columns):
    expected = a["price_vnd_million"] / a["gdp_binh_quan_dau_nguoi_trieu_dong"]
    mismatch = (expected - a["price_to_gdp_ratio"]).abs().gt(5e-4) & expected.notna()
    print("Bất nhất price_to_gdp_ratio:", int(mismatch.sum()))
if {"scraped_at", "year", "vehicle_age"}.issubset(a.columns):
    age_mismatch = (a["scraped_at"].dt.year - a["year"]).ne(a["vehicle_age"])
    print("Bất nhất vehicle_age:", int(age_mismatch.sum()))
print("ChangeLog số sự kiện:", len(log))
display(log["ly_do"].value_counts().rename_axis("Loại thay đổi").to_frame("Sự kiện"))
print("Sai lệch tuyệt đối lớn nhất (do tỷ lệ làm tròn):",float((expected - a["price_to_gdp_ratio"]).abs().max()))
print("TRẠNG THÁI NGUỒN MACRO: chưa có tài liệu xác minh nguồn, năm và định nghĩa đo.")
print("Quy tắc: KHÔNG sử dụng GDP, dân số, đô thị hóa và price_to_gdp_ratio cho các kết luận thống kê / kinh tế.")


Listings ID unique: True
Các địa phương thuộc MacroLookup: True
Bất nhất price_to_gdp_ratio: 0
Bất nhất vehicle_age: 0
ChangeLog số sự kiện: 265


,Sự kiện
Loại thay đổi,
Trùng listing_code (bản sao),198
Electric nghi nhầm (khớp chữ 'điện' trong mô tả); không đủ bằng chứng loại nhiên liệu,56
Tên phiên bản chứa HEV/Hybrid,7
Xe mới không có số km đã đi (nghi bắt nhầm '4L/100km'),3
Không phải tin đăng,1


Sai lệch tuyệt đối lớn nhất (do tỷ lệ làm tròn): 0.0004285714285714448
TRẠNG THÁI NGUỒN MACRO: chưa có tài liệu xác minh nguồn, năm và định nghĩa đo.
Quy tắc: KHÔNG sử dụng GDP, dân số, đô thị hóa và price_to_gdp_ratio cho các kết luận thống kê / kinh tế.


## 3. Descriptive Statistics — mức trung tâm, phân tán, IQR

- **Mean:** nhạy với quan sát cực lớn; **median:** đại diện tốt hơn khi lệch phải.
- **Mode:** dùng cho hãng/tỉnh/nhiên liệu phổ biến; mode số có thể ít ý nghĩa khi giá là số được làm tròn.
- **Sample variance / SD (ddof=1):** mức phân tán theo đơn vị bình phương / cùng đơn vị gốc.
- **IQR = Q3 − Q1**; ngoài `[Q1−1.5IQR, Q3+1.5IQR]` là **ứng viên kiểm tra**, không tự động là lỗi.

In [6]:
num_cols = [c for c in ["price_vnd_million", "year", "vehicle_age", "mileage_km",
    "gdp_binh_quan_dau_nguoi_trieu_dong", "ty_le_do_thi_hoa"] if c in a.columns]
desc = a[num_cols].describe(percentiles=[.25,.5,.75,.9,.95,.99]).T
desc["variance_sample"] = a[num_cols].var(ddof=1)
desc["IQR"] = desc["75%"] - desc["25%"]
desc["skewness"] = a[num_cols].skew()
for cat in ["brand", "province", "condition", "fuel_type"]:
    if cat in a.columns:
        print(f"Mode của {cat}: {a[cat].mode(dropna=True).tolist()}")
q1, q3 = a["price_vnd_million"].quantile([.25,.75])
iqr = q3-q1
upper = q3+1.5*iqr
outlier_price_n = int((a["price_vnd_million"] > upper).sum())
print("Giá IQR upper:", upper, "| n vượt upper:", outlier_price_n)
display(desc.round(3))
desc.to_csv(TABLE_DIR / "05_descriptive_statistics.csv", encoding="utf-8-sig")
by_cond = a.groupby("condition")["price_vnd_million"].agg(["count", "mean", "median", "std"])
by_brand = a.groupby("brand")["price_vnd_million"].agg(["count", "median", "mean"]).sort_values("count",ascending=False)
by_province = a.groupby("province")["price_vnd_million"].agg(["count", "median", "mean"]).sort_values("count",ascending=False)
display(by_cond.round(2), by_brand.round(2), by_province.round(2))
for fname,table in [("06_by_condition",by_cond),("07_by_brand",by_brand),("08_by_province",by_province)]:
    table.to_csv(TABLE_DIR / (fname + ".csv"), encoding="utf-8-sig")

# Mode thương hiệu bao gồm đồng hạng; giá liên tục dùng lớp histogram phổ biến nhất.
brand_modes = a["brand"].value_counts().loc[lambda s: s.eq(s.max())]
print("Mode hãng (cùng tần số cao nhất):",brand_modes.to_dict())
hist_counts, hist_edges = np.histogram(a["price_vnd_million"], bins=np.histogram_bin_edges(a["price_vnd_million"], bins="fd"))
ix_mode = int(np.argmax(hist_counts))
print(f"Lớp giá phổ biến nhất theo FD-histogram: [{hist_edges[ix_mode]:.1f}, {hist_edges[ix_mode+1]:.1f}) triệu, n={hist_counts[ix_mode]}")
price_var = float(a["price_vnd_million"].var(ddof=1))
price_sd = float(a["price_vnd_million"].std(ddof=1))
print(f"Variance mẫu: {price_var:,.2f} (triệu VND)²; SD: {price_sd:,.2f} triệu VND")
print(f"IQR = {iqr:.2f} triệu; 1.35 × SD = {1.35*price_sd:.2f} triệu (kiểm tra gần đúng nếu Normal).")


Mode của brand: ['Hyundai', 'Kia']
Mode của province: ['Hà Nội']
Mode của condition: ['cũ']
Mode của fuel_type: ['Petrol']
Giá IQR upper: 1192.5 | n vượt upper: 34


,count,mean,std,min,25%,50%,75%,90%,95%,99%,max,variance_sample,IQR,skewness
price_vnd_million,632.0,593.076,473.760,27.00,355.00,495.00,690.00,977.90,1209.00,2657.00,4730.00,2.244486e+05,335.00,4.729
year,632.0,2020.225,4.950,1993.00,2018.00,2021.00,2024.00,2026.00,2026.00,2026.00,2026.00,2.450400e+01,6.00,-1.445
vehicle_age,632.0,5.775,4.950,0.00,2.00,5.00,8.00,12.00,15.00,22.00,33.00,2.450400e+01,6.00,1.445
mileage_km,414.0,72443.519,58658.473,13.00,36000.00,66000.00,97500.00,129700.00,150000.00,342720.00,517249.00,3.440816e+09,61500.00,2.904
gdp_binh_quan_dau_nguoi_trieu_dong,632.0,128.766,46.330,60.00,100.00,140.00,150.00,200.00,200.00,200.00,200.00,2.146493e+03,50.00,-0.065
ty_le_do_thi_hoa,632.0,0.488,0.203,0.15,0.45,0.49,0.68,0.73,0.73,0.73,0.73,4.100000e-02,0.23,-0.506


,count,mean,median,std
condition,,,,
cũ,552,553.93,465.0,449.71
mới,80,863.19,719.5,545.53


,count,median,mean
brand,,,
Hyundai,100,492.0,557.64
Kia,100,444.5,512.22
Ford,98,650.5,700.43
Toyota,93,465.0,813.10
Mazda,89,495.0,522.91
Honda,81,474.0,530.40
Mitsubishi,71,495.0,479.96


,count,median,mean
province,,,
Hà Nội,142,679.0,836.54
Đồng Nai,133,379.0,402.47
Nghệ An,131,499.0,517.65
Cần Thơ,116,415.0,561.03
Quảng Ninh,110,582.5,632.85


Mode hãng (cùng tần số cao nhất): {'Hyundai': 100, 'Kia': 100}
Lớp giá phổ biến nhất theo FD-histogram: [412.5, 489.6) triệu, n=89
Variance mẫu: 224,448.57 (triệu VND)²; SD: 473.76 triệu VND
IQR = 335.00 triệu; 1.35 × SD = 639.58 triệu (kiểm tra gần đúng nếu Normal).


## 4. Normalization — minh họa, không làm sai lệch EDA

Báo cáo vẫn dùng **triệu VND, năm, km** trên trục gốc. Z-score `(x−mean)/SD` và Min–Max `(x−min)/(max−min)` chỉ lập bảng minh họa; chúng không giải quyết outlier, không được tự áp dụng ngược vào dữ liệu nguồn. Khi dựng mô hình dự báo sau này phải `fit` scaler trên **train** và `transform` validation/test để tránh leakage.

**Bổ sung:** áp dụng `log(price)` cho **toàn bộ 632 giá dương**, rồi đối chiếu skewness và Shapiro để xác nhận log chỉ giúp đối xứng hơn, không tự động tạo Normal. Bảng chuẩn hóa đầy đủ được xuất CSV; không chỉ chạy 5 dòng ví dụ. Không sử dụng `year` đồng thời với `vehicle_age` khi đo tương quan.

In [7]:
scaler_demo = pd.DataFrame(index=a.index)
for c in ["price_vnd_million", "vehicle_age", "mileage_km"]:
    s = a[c].astype(float)
    sigma = s.std(ddof=1)
    span = s.max()-s.min()
    scaler_demo[c + "_zscore"] = (s-s.mean())/sigma if sigma and np.isfinite(sigma) else np.nan
    scaler_demo[c + "_minmax"] = (s-s.min())/span if span and np.isfinite(span) else np.nan
print("Bảng minh họa chuẩn hóa (5 dòng, không thay đổi dữ liệu gốc):")
display(scaler_demo.head())
scaler_demo.head(100).to_csv(TABLE_DIR / "09_normalization_demo_first100.csv",index=False,encoding="utf-8-sig")

# Tính transform cho toàn bộ 632 dòng; giữ original không đổi.
a["log_price"] = np.log(a["price_vnd_million"].astype(float))
scaler_demo["log_price"] = a["log_price"]
scaler_demo.to_csv(TABLE_DIR / "10_normalization_all_rows.csv", index_label="source_index", encoding="utf-8-sig")
print("Bảng biến đổi đầy đủ (n):",len(scaler_demo),"| skew gốc:",round(a.price_vnd_million.skew(),4),"| skew log:",round(a.log_price.skew(),4))


Bảng minh họa chuẩn hóa (5 dòng, không thay đổi dữ liệu gốc):


,price_vnd_million_zscore,price_vnd_million_minmax,vehicle_age_zscore,vehicle_age_minmax,mileage_km_zscore,mileage_km_minmax
0,0.647847,0.185626,-1.166692,0.000000,NaN,NaN
1,1.236753,0.244950,0.853443,0.303030,0.810735,0.231977
2,-0.333662,0.086753,0.045389,0.181818,0.469778,0.193310
3,0.204585,0.140974,0.449416,0.242424,-0.962240,0.030909
4,1.386618,0.260047,-0.964679,0.030303,-0.536044,0.079242


Bảng biến đổi đầy đủ (n): 632 | skew gốc: 4.7289 | skew log: -0.1082


## 4.1. Phân phối chuẩn, T-distribution và kiểm định đơn giản (học liệu 6.1–6.4)

- **Normal:** so IQR với 1,35 SD (chỉ gần đúng với dữ liệu lý tưởng chuẩn); kiểm tra hình dạng bằng Q–Q plot và **Shapiro–Wilk**. Khi p nhỏ, bác bỏ giả thuyết dữ liệu chuẩn trong khuôn khổ phép thử; mẫu lớn dễ phát hiện sai lệch nhỏ.
- **Xác suất chuẩn:** tính xác suất giá trên 1.000 triệu nếu *giả sử* Normal với mean/SD quan sát rồi so với tỷ lệ thực tế. Đây là **minh họa phản ví dụ**, không phải dự báo xác suất thị trường.
- **t-distribution:** tính *khoảng tin cậy t mang tính minh họa* cho **trung bình log-giá** và nêu rõ hạn chế do log-giá vẫn không hoàn toàn chuẩn và mẫu không ngẫu nhiên; không sử dụng t-test để so hai nhóm giá gốc lệch mạnh.
- **Spearman:** đo liên hệ **đơn điệu theo thứ hạng** giữa tuổi xe và giá trong xe cũ, khác Pearson đo tuyến tính trên dữ liệu gốc.
- **Mann–Whitney U:** mô tả kiểm định xếp hạng hai nhóm độc lập, không diễn giải mặc định thành hiệu ứng nhân quả hay kiểm định trung vị khi hình dạng phân phối khác nhau.

**Phần mở rộng đã bỏ khỏi báo cáo chính:** bootstrap CI, rank-biserial, epsilon-squared, Kruskal–Wallis, giúp giảm kỹ thuật vượt phạm vi giảng dạy.

> Vì giá sau log vẫn không chuẩn theo Shapiro–Wilk, phép ước lượng khoảng tin cậy t chỉ nhằm minh họa nội dung bài giảng. Không dùng khoảng này làm phát biểu về trung bình giá của toàn thị trường.


In [8]:
# Kiểm tra Normal ở giá gốc và log-giá; không ngầm coi log-giá là chuẩn.
from scipy.stats import norm, t, shapiro
raw_shapiro = shapiro(a["price_vnd_million"].to_numpy())
log_shapiro = shapiro(a["log_price"].to_numpy())
normal_theoretical = float(norm.sf(1000,loc=a.price_vnd_million.mean(),scale=a.price_vnd_million.std(ddof=1)))
normal_observed = float((a.price_vnd_million > 1000).mean())
print(f"SHAPIRO giá: W={raw_shapiro.statistic:.4f}, p={raw_shapiro.pvalue:.3e}")
print(f"SHAPIRO log-giá: W={log_shapiro.statistic:.4f}, p={log_shapiro.pvalue:.3e}")
print(f"P(Giá>1000) nếu ép Normal: {normal_theoretical:.2%} | tần suất mẫu: {normal_observed:.2%}")
print("Điều kiện Normal không phù hợp; tần suất mẫu cũng không là xác suất đại diện quốc gia.")

# 95% t-interval của mean(log-price): minh họa công thức bài học, KHÔNG gọi là CI đại diện quốc gia.
log_vals=a["log_price"].to_numpy()
nlog=len(log_vals)
log_mean=float(log_vals.mean()); log_sd=float(log_vals.std(ddof=1))
tcrit=float(t.ppf(.975,df=nlog-1))
log_mean_ci=(log_mean-tcrit*log_sd/np.sqrt(nlog),log_mean+tcrit*log_sd/np.sqrt(nlog))
print(f"t_{nlog-1}(.975)={tcrit:.4f}; minh họa 95% CI mean(log-price): {log_mean_ci}")

used = a.loc[a.condition=="cũ"].copy()
new = a.loc[a.condition=="mới"].copy()
sp_age = stats.spearmanr(used["vehicle_age"],used["price_vnd_million"],nan_policy="omit")
mw = stats.mannwhitneyu(used["price_vnd_million"],new["price_vnd_million"],alternative="two-sided")
used_known_km = used.loc[used.mileage_km.notna(),"price_vnd_million"]
used_missing_km = used.loc[used.mileage_km.isna(),"price_vnd_million"]
mw_missing = stats.mannwhitneyu(used_missing_km,used_known_km,alternative="two-sided")
sp_km = stats.spearmanr(used.loc[used.mileage_km.notna(),"mileage_km"],used_known_km)

stat_tests = pd.DataFrame([
    ["Shapiro giá gốc",len(a),raw_shapiro.statistic,raw_shapiro.pvalue,"Dữ liệu không phù hợp Normal"],
    ["Shapiro log-giá",len(a),log_shapiro.statistic,log_shapiro.pvalue,"Log giảm skew nhưng chưa Normal"],
    ["Spearman tuổi–giá (xe cũ)",len(used),sp_age.statistic,sp_age.pvalue,"Tương quan thứ hạng; không nhân quả"],
    ["Spearman km–giá (xe cũ đủ km)",len(used_known_km),sp_km.statistic,sp_km.pvalue,"Bị ảnh hưởng bởi dữ liệu km thiếu"],
    ["Mann–Whitney (xe mới vs xe cũ)",len(a),mw.statistic,mw.pvalue,"So sánh thứ hạng, không khẳng định nhân quả"],
    ["Mann–Whitney (thiếu vs đủ km; xe cũ)",len(used),mw_missing.statistic,mw_missing.pvalue,"Thăm dò missingness; có biến nhiễu"],
],columns=["Kiểm định","n mẫu liên quan","statistic","p_value","Giải thích"])
display(stat_tests)
stat_tests.to_csv(TABLE_DIR / "12_tests_core.csv",index=False,encoding="utf-8-sig")


SHAPIRO giá: W=0.6199, p=4.706e-35
SHAPIRO log-giá: W=0.9714, p=9.202e-10
P(Giá>1000) nếu ép Normal: 19.52% | tần suất mẫu: 9.49%
Điều kiện Normal không phù hợp; tần suất mẫu cũng không là xác suất đại diện quốc gia.
t_631(.975)=1.9637; minh họa 95% CI mean(log-price): (np.float64(6.152334386547781), np.float64(6.245545934616158))


,Kiểm định,n mẫu liên quan,statistic,p_value,Giải thích
0,Shapiro giá gốc,632,0.619858,4.706187e-35,Dữ liệu không phù hợp Normal
1,Shapiro log-giá,632,0.971419,9.201845e-10,Log giảm skew nhưng chưa Normal
2,Spearman tuổi–giá (xe cũ),552,-0.602309,8.524665e-56,Tương quan thứ hạng; không nhân quả
3,Spearman km–giá (xe cũ đủ km),414,-0.238331,9.317763e-07,Bị ảnh hưởng bởi dữ liệu km thiếu
4,Mann–Whitney (xe mới vs xe cũ),632,9622.500000,3.295636e-16,"So sánh thứ hạng, không khẳng định nhân quả"
5,Mann–Whitney (thiếu vs đủ km; xe cũ),552,11203.500000,1.013666e-26,Thăm dò missingness; có biến nhiễu
